# FaST-ISL: train and evaluate on iSign (Colab GPU)

**Before this notebook:** run `isign_convert_poses.ipynb` once. It writes the iSign keypoint shards to
`MyDrive/ISL/isign_pose_keypoints/` (about 127k clips).

Use a GPU runtime (Runtime → Change runtime type → T4 GPU or better).
Every run saves to Drive after each epoch. If Colab disconnects, run all cells again: finished runs
are skipped and the current one continues from its last epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf ISL && git clone -q https://github.com/sengarshivansh/ISL.git
%cd /content/ISL
!pip -q install sacrebleu rouge-score sentencepiece pyyaml

## 1. Check the keypoint shards

In [ ]:
import glob, json
SHARDS = '/content/drive/MyDrive/ISL/isign_pose_keypoints'
files = sorted(glob.glob(f'{SHARDS}/shard_*.npz'))
print(len(files), 'shards found')
assert files, 'No shards: run isign_convert_poses.ipynb first'
print(json.load(open(f'{SHARDS}/layout.json'))['feat_dim'], 'values per frame (expected 714)')

## 2. Quick check (10 minutes)
Trains the full model for one epoch on 5,000 clips to prove everything works on the real data.

In [ ]:
!python -m islt.train --config configs/isign_base.yaml --set name=smoke data.limit=5000 train.epochs=1 \
    out_dir=/content/drive/MyDrive/ISL/runs_smoke eval.speeds='[1.0]'

## 3. All experiments
Runs every config in `configs/ablations/` (9 runs). On a T4 GPU each takes a few hours, so
this needs several Colab sessions; re-run this cell each time. To get the headline
result first, it runs the baseline and the full model before the region ablations.

In [ ]:
import os, subprocess
RUNS = '/content/drive/MyDrive/ISL/runs'
order = ['B0_body_hands', 'FaST', 'B1_speedaug', 'FaST_noaug', 'FaST_concat',
         'F_mouth', 'F_eyebrows', 'F_eyes', 'F_head']
for name in order:
    if os.path.exists(f'{RUNS}/{name}/results.json'):
        print('done:', name); continue
    print('=== training', name, flush=True)
    code = f'''
import yaml
from islt.train import load_config, deep_update, train
cfg = load_config("configs/isign_base.yaml")
deep_update(cfg, yaml.safe_load(open("configs/ablations/{name}.yaml")))
cfg["out_dir"] = "{RUNS}"
train(cfg)
'''
    subprocess.run(['python', '-c', code], check=True)

## 4. Results table

In [ ]:
!python -m islt.report /content/drive/MyDrive/ISL/runs --out /content/drive/MyDrive/ISL/results_table.tex

## 5. Sentence-boundary detector (for continuous, fast signing)
Trained on stitched training streams with random speed (0.5–2×) and random pauses (often none). About 20 minutes on GPU.

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive/ISL/runs/boundary/boundary.pt'):
    !python -m islt.boundary --config configs/isign_base.yaml --out /content/drive/MyDrive/ISL/runs/boundary --steps 6000 --device cuda

## 6. Continuous-signing evaluation (streams of 5 sentences)
Pause-based vs learned segmenter, at 0.8 s / 0.4 s / no pause and 1× / 1.5× / 2× speed.

In [ ]:
R = '/content/drive/MyDrive/ISL/runs'
for run in ['FaST', 'B0_body_hands']:
    !python -m islt.eval_stream --config configs/isign_base.yaml --checkpoint {R}/{run}/best.pt
    !python -m islt.eval_stream --config configs/isign_base.yaml --checkpoint {R}/{run}/best.pt --boundary {R}/boundary/boundary.pt

## 7. Bring results back
Download `MyDrive/ISL/runs/*/results.json`, `stream_results.json` and `FaST/best.pt` into
`ISL/runs/` on the Mac. Then `python -m islt.report runs` rebuilds the paper tables, and
`ISL_CHECKPOINT=runs/FaST/best.pt python -m app.server` runs the demo with the real model.
Also copy `runs/boundary/boundary.pt` and start the demo with `ISL_BOUNDARY=runs/boundary/boundary.pt`.